# NeurIPS 2026 Smart Buildings — Submission Version 5

**Experiment:** Regret-aware HVAC allocation and slow online actuation calibration. This is a new control family relative to V2/V3 (per-AHU PID) and V4 (trend forecasts with supervisory hysteresis). V5 has NOT been scored on Kaggle. Lower is better.

## Architecture
- Direct per-zone sensor-band excursions, increasing convex tail-risk surrogate, and a whole-AHU cooling/heating allocation decision that preserves outlier demand.
- AHUs cool only; VAV reheat and the building's hot-water plant handle heating. Cold zones on mixed-demand AHUs receive restricted airflow rather than full cooling airflow.
- Fan pressure and hot-water reset allocated by 75th-percentile demand, active-zone coverage, and bounded worst-zone corrections; economizer considers exhaust fan power.
- Per-AHU strength [0.8, 1.4] adapts after 30 minutes of persistent unchanged mode, based on measured comfort-error reduction (no forecast, PID gains, or saved model).
- Uses each zone's current scoring setpoint sensors and a schedule fallback; true per-zone occupancy counts are not exposed, so no fabricated occupancy weights.

## Submission and tests
The self-contained notebook uses the official Kaggle inference-server interface: TransferPolicy.__init__, reset, get_action; produces float32 continuous actions in [-1,1] and int32 discrete actions in {0,1}. CPU-only, no internet, no external artifacts. Run the FINAL cell in Kaggle to start the inference server.

The flags ADAPT, RISK_WEIGHTING, ECONOMIZER support controlled ablations. The repository's test_version5.py is an offline synthetic contract test of the exact notebook policy code (variable shapes, metadata indexing, mixed hot/cold, physical actions, economizer, calibration, episode reset). It is not a 30-day public simulator evaluation. Run the official public scenarios before claiming improvement.

In [ ]:
"""V5: comfort-risk allocation with online actuator-strength calibration.

No PID, weather memorization, model weights, or look-ahead forecast.  Every
parameter is shared across buildings, and all device locations come from the
current episode's public metadata.
"""
import re
import numpy as np
from kaggle_evaluation import smart_buildings_inference_server


class TransferPolicy(smart_buildings_inference_server.Policy):
    """Allocate HVAC effort by per-zone comfort risk and observed response.

    The AHU only cools.  Every heating command is therefore implemented with
    VAV reheat and the shared hot-water system.  Unlike average-zone PID, an
    outlying zone cannot disappear in the mean; unlike V4, no temperature
    forecast is required.  A bounded six-step response check adjusts the
    actuation strength to buildings with different thermal envelopes.
    """

    ADAPT = True  # Ablation: set False to fix per-AHU strength at 1.0.
    RISK_WEIGHTING = True  # Ablation: False uses raw K excursions.
    ECONOMIZER = True

    def __init__(self, building_meta_data, action_spec):
        super().__init__(building_meta_data, action_spec)
        m = building_meta_data
        self.n_cont = int(m['n_continuous_action'])
        self.n_disc = int(m['n_discrete_action'])
        self.temp_actions = list(map(int, m['temp_indices']))
        self.press_actions = list(map(int, m['pressure_indices']))
        self.recirc_actions = list(map(int, m['recirculation_indices']))
        self.water_actions = list(map(int, m['water_indices']))
        self.n_ahu = len(self.temp_actions)
        self.water_temp = self.water_actions[0] if self.water_actions else None
        self.water_pressure = (self.press_actions[self.n_ahu]
                               if len(self.press_actions) > self.n_ahu else None)
        self.water_run = self.n_disc - 1
        self.fields = {name: i for i, name in enumerate(m['observation_field_names'])}
        self.comfort = tuple(map(float, m['comfort_temp_window']))
        self.eco = tuple(map(float, m['eco_temp_window']))
        self.morning = float(m['morning_start_hour'])
        self.evening = float(m['evening_start_hour'])
        norm = m.get('observation_normalization_metadata', {}).get(
            'zone_air_temperature_sensor', {})
        self.zone_mean = float(norm.get('sample_mean', 0.0))
        self.zone_std = float(norm.get('sample_std', 1.0))
        if not np.isfinite(self.zone_std) or self.zone_std <= 0.0:
            self.zone_std = 1.0

        def order(name):
            match = re.search(r'\d+', str(name))
            return (int(match.group()) if match else -1, str(name))

        assignments = m['ahu_zone_assignments']
        zone_action = m['zone_action_indices']
        self.ahus = []
        for i, ahu_id in enumerate(sorted(assignments, key=order)[:self.n_ahu]):
            zones = []
            for room in assignments[ahu_id]:
                vav = str(room).replace('room_', 'VAV_')
                temp = self.fields.get(vav + '_zone_air_temperature_sensor')
                if temp is None:
                    continue
                actions = zone_action.get(vav, {})
                zones.append({
                    'temp': temp,
                    'lo': self.fields.get(vav + '_zone_air_heating_temperature_setpoint_sensor'),
                    'hi': self.fields.get(vav + '_zone_air_cooling_temperature_setpoint_sensor'),
                    'damper': actions.get('supply_air_damper_percentage_command'),
                    'valve': actions.get('reheat_valve_percentage_command'),
                })
            self.ahus.append({
                'zones': zones,
                'run': i,
                'supply': self.temp_actions[i],
                'pressure': self.press_actions[i] if i < len(self.press_actions) else None,
                'recirc': self.recirc_actions[i] if i < len(self.recirc_actions) else None,
                'outdoor': self.fields.get(f'ahs_{ahu_id}_outside_air_temperature_sensor'),
            })
        self.reset()

    def reset(self):
        """The server may reuse a policy over episodes; never leak state."""
        self.strength = np.ones(len(self.ahus), dtype=np.float64)
        self.mode = ['off'] * len(self.ahus)
        self.mode_age = [0] * len(self.ahus)
        self.reference_severity = [0.0] * len(self.ahus)

    @staticmethod
    def _fraction(value):
        return float(np.clip(value, 0.0, 1.0))

    @classmethod
    def _command(cls, fraction):
        return 2.0 * cls._fraction(fraction) - 1.0

    def _schedule_band(self, obs):
        def phase(s, c, period):
            a, b = self.fields.get(s), self.fields.get(c)
            if a is None or b is None:
                return None
            x, y = float(obs[a]), float(obs[b])
            if not (np.isfinite(x) and np.isfinite(y)):
                return None
            return float(np.arctan2(x, y) % (2.0 * np.pi)) * period / (2.0 * np.pi)

        hour = phase('hod_sin_000', 'hod_cos_000', 24.0)
        day = phase('dow_sin_000', 'dow_cos_000', 7.0)
        occupied = (hour is not None and day is not None and
                    int(np.floor(day + 1e-6)) < 5 and
                    self.morning <= hour < self.evening)
        return self.comfort if occupied else self.eco

    def _read_zone(self, obs, zone, fallback):
        temp = float(obs[zone['temp']]) * self.zone_std + self.zone_mean
        if not (np.isfinite(temp) and 240.0 <= temp <= 340.0):
            return None
        lo_idx, hi_idx = zone['lo'], zone['hi']
        lo = float(obs[lo_idx]) if lo_idx is not None else float('nan')
        hi = float(obs[hi_idx]) if hi_idx is not None else float('nan')
        if not (np.isfinite(lo) and np.isfinite(hi) and
                245.0 <= lo < hi <= 335.0 and hi - lo <= 35.0):
            lo, hi = fallback
        return {'zone': zone, 'temp': temp, 'lo': lo, 'hi': hi,
                'hot': max(0.0, temp - hi), 'cold': max(0.0, lo - temp)}

    def _risk(self, excursion):
        """A smooth surrogate of the scorer's unbounded tail comfort loss."""
        if not self.RISK_WEIGHTING:
            return excursion
        return excursion + max(0.0, excursion - 5.0) ** 2 / 5.0

    def _calibrate(self, i, mode, severity):
        """Bounded, six-step response calibration, not a PID integrator.

        Compare the current violation against its value 30 minutes earlier,
        *only* while the same mode persists. This is intentionally slow to
        avoid reacting to one noisy sensor reading or one cooling transition.
        """
        if mode == 'off' or mode != self.mode[i]:
            self.mode[i] = mode
            self.mode_age[i] = 0
            self.reference_severity[i] = severity
            return
        self.mode_age[i] += 1
        if self.mode_age[i] < 6:
            return
        improvement = self.reference_severity[i] - severity
        if self.ADAPT and severity >= 0.5:
            if improvement < 0.02:
                self.strength[i] = min(1.40, self.strength[i] * 1.04)
            elif improvement > 0.25:
                self.strength[i] = max(0.80, self.strength[i] * 0.97)
        self.reference_severity[i] = severity
        self.mode_age[i] = 0

    def _economize(self, outdoor, indoor, target, pressure_pa):
        # AHU recirculation is physically capped at 70%; outside air is NOT free
        # because the exhaust fan consumes electricity.
        if (not self.ECONOMIZER or not np.isfinite(outdoor) or
                outdoor >= indoor or indoor - outdoor <= pressure_pa * 0.00322):
            return 1.0
        useful_fraction = (indoor - max(target, 285.0)) / max(1e-6, indoor - outdoor)
        return self._command(1.0 - float(np.clip(useful_fraction, 0.30, 1.0)))

    def _initial_actions(self):
        """Allocate one action vector for this building, using safe idle defaults.

        Continuous actions are normalized to [-1, 1]. Zero is the midpoint
        of a physical actuator range, *not* an off command. A supply-air
        command of +1 requests the warmest supply (no mechanical cooling).
        """
        continuous = np.full(self.n_cont, -1.0, dtype=np.float32)
        discrete = np.zeros(self.n_disc, dtype=np.int32)
        for index in self.temp_actions:
            continuous[index] = 1.0
        for index in self.recirc_actions:
            continuous[index] = 1.0
        return continuous, discrete

    def _zone_statuses(self, obs, ahu, fallback_band):
        """Read valid zone temperatures and their current comfort violations.

        Each status holds 'hot' and 'cold' in Kelvin; at most one is nonzero.
        Missing or implausible temperature sensors are skipped. Setpoint
        sensors are preferred to the fallback weekday/comfort schedule.
        """
        statuses = [self._read_zone(obs, zone, fallback_band)
                    for zone in ahu['zones']]
        return [status for status in statuses if status is not None]

    def _select_mode(self, zones):
        """Choose off, cool, or reheat for an AHU serving several VAV zones.

        One AHU has only one supply-air temperature. We combine total zone
        comfort risk with an extra worst-hot-zone term, so severe overheating
        cannot disappear behind the average. The AHU cannot heat; 'heat'
        means airflow plus hot-water reheat at each cold VAV.

        Returns (mode, severity_K). No PID state or look-ahead is involved.
        """
        hottest = max(zone['hot'] for zone in zones)
        coldest = max(zone['cold'] for zone in zones)
        if max(hottest, coldest) <= 0.12:
            return 'off', 0.0

        heating_risk = sum(self._risk(zone['cold']) for zone in zones)
        cooling_risk = sum(self._risk(zone['hot']) for zone in zones)
        cooling_wins = (hottest > 0.12 and
                        (cooling_risk + 0.7 * self._risk(hottest) >=
                         0.65 * heating_risk or hottest >= 2.5))
        return ('cool', hottest) if cooling_wins else ('heat', coldest)

    def _set_fan_pressure(self, ahu, zones, mode, severity, gain, continuous):
        """Spend fan power in proportion to demand, not full blast by default.

        The 75th-percentile active-mode violation reflects typical need;
        the worst violation retains some influence. The fraction of zones
        >0.3 K outside their band reduces fan pressure when just one zone
        needs help. Returns (pressure_Pa, p75_excursion_K).
        """
        if mode == 'cool':
            demands = [zone['hot'] for zone in zones]
        else:
            demands = [zone['cold'] for zone in zones]

        fraction_demanding = sum(d > 0.3 for d in demands) / len(demands)
        representative = float(np.percentile(demands, 75))
        severity_fraction = self._fraction(
            gain * (representative + 0.35 * severity) / 3.2)
        pressure_pa = 200.0 + 1450.0 * severity_fraction * (
            0.40 + 0.60 * np.sqrt(fraction_demanding))
        pressure_pa = float(np.clip(pressure_pa, 200.0, 1800.0))

        if ahu['pressure'] is not None:
            continuous[ahu['pressure']] = float(np.clip(
                pressure_pa / 1000.0 - 1.0, -1.0, 1.0))
        return pressure_pa, representative

    def _set_cooling(self, obs, ahu, zones, gain, representative,
                     hottest, pressure_pa, continuous):
        """Set cooling supply air and optional outdoor-air economizer.

        AHU supply air ranges from 285 to 305 K. Heating uses the warmest
        command, assigned by _initial_actions; this method runs only when
        the mode is 'cool'. An economizer is useful only when its chiller
        savings exceed the incremental exhaust-fan work.
        """
        target = float(np.clip(
            294.0 - 3.0 * gain * (0.65 * representative + 0.35 * hottest),
            285.0, 296.0))
        continuous[ahu['supply']] = float(np.clip(
            (target - 285.0) / 10.0 - 1.0, -1.0, 1.0))

        outside_index = ahu['outdoor']
        outside = float(obs[outside_index]) if outside_index is not None else np.nan
        inside = float(np.mean([zone['temp'] for zone in zones]))
        if ahu['recirc'] is not None:
            continuous[ahu['recirc']] = self._economize(
                outside, inside, target, pressure_pa)

    def _set_vav_boxes(self, zones, mode, gain, continuous, water_demands):
        """Allocate per-zone dampers and VAV reheat valves.

        In cooling mode: hot zones get cooling airflow, while cold zones get
        only 18% of their heating airflow request to carry reheat. In heating
        mode: cold zones get proportional airflow. A cold VAV can reheat even
        while its shared AHU is cooling other zones.

        water_demands receives (cold_shortfall_K, valve_open_fraction) only
        for VAVs that actually request heat and have a reheat-valve action.
        """
        for status in zones:
            zone = status['zone']
            hot, cold = status['hot'], status['cold']
            cooling_fraction = (self._fraction(gain * (hot + 0.05) / 2.4)
                                if mode == 'cool' else 0.0)
            heating_fraction = self._fraction(gain * (cold + 0.05) / 2.6)
            heating_airflow = (0.18 * heating_fraction
                               if mode == 'cool' else heating_fraction)
            airflow_fraction = max(cooling_fraction, heating_airflow)

            if zone['damper'] is not None:
                continuous[zone['damper']] = self._command(airflow_fraction)

            if cold > 0.12 and zone['valve'] is not None:
                valve_fraction = self._fraction(gain * (cold + 0.10) / 2.2)
                continuous[zone['valve']] = self._command(valve_fraction)
                water_demands.append((cold, valve_fraction))

    def _set_hot_water(self, water_demands, continuous, discrete):
        """Switch on the shared boiler/pump only when some VAV needs reheat.

        The 75th-percentile cold violation is robust to one outlying zone;
        a bounded worst-zone supplement guards against persistent severe
        deficits. Maps the requested 320..348 K to the plant's 310..350 K
        action range and sets pump pressure from mean valve opening.
        """
        if not water_demands or not (0 <= self.water_run < self.n_disc):
            return

        discrete[self.water_run] = 1
        shortfalls = [shortfall for shortfall, _ in water_demands]
        valve_load = [valve for _, valve in water_demands]
        p75 = float(np.percentile(shortfalls, 75))
        water_fraction = self._fraction(
            p75 / 2.8 + 0.15 * max(shortfalls) / 4.0)
        water_temperature_k = 320.0 + 28.0 * water_fraction

        if self.water_temp is not None:
            continuous[self.water_temp] = self._command(
                (water_temperature_k - 310.0) / 40.0)
        if self.water_pressure is not None:
            continuous[self.water_pressure] = self._command(
                max(0.38, 0.35 + 0.65 * float(np.mean(valve_load))))

    def get_action(self, obs):
        """Run one five-minute control decision and return Kaggle actions.

        Pipeline: observation -> per-zone deviations -> per-AHU operating mode
        -> fan and cooling commands -> per-VAV dampers/reheat -> shared water.
        The 30-minute online calibration is updated before action allocation.
        """
        obs = np.asarray(obs, dtype=np.float64).reshape(-1)
        continuous, discrete = self._initial_actions()
        fallback_band = self._schedule_band(obs)
        water_demands = []

        for index, ahu in enumerate(self.ahus):
            zones = self._zone_statuses(obs, ahu, fallback_band)
            if not zones:
                self._calibrate(index, 'off', 0.0)
                continue

            mode, severity = self._select_mode(zones)
            if mode == 'off':
                self._calibrate(index, 'off', 0.0)
                continue

            self._calibrate(index, mode, severity)
            gain = float(self.strength[index])
            discrete[ahu['run']] = 1

            pressure_pa, representative = self._set_fan_pressure(
                ahu, zones, mode, severity, gain, continuous)
            if mode == 'cool':
                self._set_cooling(obs, ahu, zones, gain, representative,
                                  severity, pressure_pa, continuous)
            self._set_vav_boxes(zones, mode, gain, continuous, water_demands)

        self._set_hot_water(water_demands, continuous, discrete)
        np.nan_to_num(continuous, copy=False, nan=-1.0, posinf=1.0, neginf=-1.0)
        np.clip(continuous, -1.0, 1.0, out=continuous)
        return {'continuous_action': continuous, 'discrete_action': discrete}


In [ ]:
# Kaggle executes the public smoke rollout and hidden evaluation.
smart_buildings_inference_server.SmartBuildingsInferenceServer(TransferPolicy).run()